In [ ]:
#Create the database and raw table
import sqlite3
conn=sqlite3.connect("orders_database.db")
conn.execute("DROP TABLE IF EXISTS orders_raw")
conn.execute("""
CREATE TABLE orders_raw(
orderID INTEGER,
CustomerName TEXT,
Email TEXT,
Phone TEXT,
City TEXT,
OrderDate TEXT,
Amount TEXT,
Status TEXT
)
""")
conn.commit()
import pandas as pd
df=pd.read_excel("/content/orders_raw.xlsx")
print("Excel file imported successfully")
data=list(df.itertuples(index=False,name=None))
cursor=conn.cursor()
cursor.executemany("""
INSERT INTO orders_raw
VALUES (?,?,?,?,?,?,?,?)
""", data)
conn.commit()
cursor.execute("""
SELECT COUNT (*)
FROM orders_raw;
""")
total=cursor.fetchone()[0]
print("Total Records:",total)
cursor=conn.execute("""
SELECT *
FROM orders_raw
LIMIT 10
""")
rows=cursor.fetchall()
for row in rows:
  print(row)
cursor=conn.execute("""
SELECT COUNT(*) AS Total_Rows
FROM orders_raw
""")
result=cursor.fetchone()
print("Total Rows:",result[0])
cursor=conn.execute("""
SELECT
     OrderID,
     COUNT(*) AS Duplicate_Count
FROM orders_raw
GROUP BY "OrderID"
HAVING COUNT(*) >1
ORDER BY Duplicate_Count DESC
""")
rows=cursor.fetchall()
if rows:
  for row in rows:
    print(row)
else:
  print("No Duplicate OrderID Found")
cursor = conn.execute("""
SELECT
    SUM(CASE WHEN "CustomerName" IS NULL OR TRIM("CustomerName") = '' THEN 1 ELSE 0 END) AS Missing_CustomerName,

    SUM(CASE WHEN "Email" IS NULL OR TRIM("Email") = '' THEN 1 ELSE 0 END) AS Missing_Email,

    SUM(CASE WHEN "Phone" IS NULL OR TRIM("Phone") = '' THEN 1 ELSE 0 END) AS Missing_Phone,

    SUM(CASE WHEN "City" IS NULL OR TRIM("City") = '' THEN 1 ELSE 0 END) AS Missing_City,

    SUM(CASE WHEN "OrderDate" IS NULL OR TRIM("OrderDate") = '' THEN 1 ELSE 0 END) AS Missing_OrderDate,

    SUM(CASE WHEN "Amount" IS NULL OR TRIM("Amount") = '' THEN 1 ELSE 0 END) AS Missing_Amount,

    SUM(CASE WHEN "Status" IS NULL OR TRIM("Status")='' THEN 1 ELSE 0 END) AS Missing_Status

FROM orders_raw
""")

result = cursor.fetchone()

print("Missing Values Report:")
print(result)
cursor = conn.execute("""
SELECT COUNT(*) - COUNT(DISTINCT "Order ID")
FROM orders_raw
""")

print("Duplicate OrderID Count:", cursor.fetchone()[0])
conn.execute("DROP TABLE IF EXISTS orders_staging")

conn.execute("""
CREATE TABLE orders_staging AS
SELECT *
FROM orders_raw
""")

conn.commit()

print("Staging Table Created Successfully")
cursor.execute("""
DELETE FROM orders_staging
WHERE rowid NOT IN (
    SELECT MIN(rowid)
    FROM orders_staging
    GROUP BY
        OrderID,
        CustomerName,
        Email,
        Phone,
        City,
        OrderDate,
        Amount,
        Status
)
""")

conn.commit()

print("Exact Duplicate Rows Removed Successfully")

Excel file imported successfully
Total Records: 10000
(3105, 'Jaya Verma', 'jaya.verma801@mail.com', '733-307-7375', ' delhi', '2024-01-05', '3091.09', 'Pending')
(6354, 'Farhan Yadav', 'farhan.yadav65@company.co.in', '23114 80745', 'Bengaluru', '04-14-2023', '1662.89', 'Refunded')
(8690, '  Chitra Reddy ', 'chitra.reddy695company.co.in', '379-490-1038', ' pune', '2024-08-03', '1610.32', 'Refunded')
(5858, ' Sneha Iyer', 'sneha.iyer945@company.co.in', '+91 8472931687', ' Coimbatore  ', '11-08-2023', '2265.32', 'Pending')
(6012, '  kabir kumar  ', 'kabir.kumar305@outlook.com', '0400977061', ' Coimbatore ', '11-19-2024', '2718.23', 'Cancelled')
(2653, 'Kabir Joshi ', 'kabir.joshi17@mail.com', '75656 19112', '  Delhi', '02-05-2023', '₹434.17', 'Pending')
(5130, 'Deepa Chatterjee  ', 'deepa.chatterjee695@outlook.com', '+91 1238943682', 'CHENNAI ', '12-02-2024', '2333.70 ', 'Refunded')
(416, 'Esha Iyer', 'esha.iyer391@gmail.com', '76927 43811', ' HYDERABAD ', '02-28-2025', '3818.14', 'Pendi

In [ ]:
# Check Missing value
cursor = conn.execute("""
SELECT COUNT(*)
FROM orders_raw
WHERE "CustomerName" IS NULL
""")

print("Missing Customer Names:", cursor.fetchone()[0])

Missing Customer Names: 199


In [ ]:
# Check Missing value

cursor = conn.execute("""
SELECT COUNT(*)
FROM orders_raw
WHERE "Email" IS NULL
""")

print("Missing Emails:", cursor.fetchone()[0])

Missing Emails: 637


In [ ]:
# Check Missing value

cursor = conn.execute("""
SELECT COUNT(*)
FROM orders_raw
WHERE "Phone" IS NULL
""")

print("Missing Phones:", cursor.fetchone()[0])

Missing Phones: 513


In [ ]:
# Check Missing value

cursor = conn.execute("""
SELECT COUNT(*)
FROM orders_raw
WHERE "City" IS NULL
""")

print("Missing Cities:", cursor.fetchone()[0])

Missing Cities: 1011


In [ ]:
# Check Missing value

cursor = conn.execute("""
SELECT COUNT(*)
FROM orders_raw
WHERE "OrderDate" IS NULL
""")

print("Missing Order Dates:", cursor.fetchone()[0])

Missing Order Dates: 329


In [ ]:
# Check Missing value

cursor = conn.execute("""
SELECT COUNT(*)
FROM orders_raw
WHERE "Amount" IS NULL
""")

print("Missing Amounts:", cursor.fetchone()[0])

Missing Amounts: 255


In [ ]:
# Check Missing value

cursor = conn.execute("""
SELECT COUNT(*)
FROM orders_raw
WHERE "Status" IS NULL
""")

print("Missing Status:", cursor.fetchone()[0])

Missing Status: 839


In [ ]:
#Remove exact duplicate rows
cursor = conn.execute("""
DELETE FROM orders_staging
WHERE rowid NOT IN (
    SELECT MIN(rowid)
    FROM orders_staging
    GROUP BY
        OrderID,
        CustomerName,
        Email,
        Phone,
        City,
        OrderDate,
        Amount,
        Status
)
""")

conn.commit()

print("Duplicate rows removed:", cursor.rowcount)

cursor = conn.execute("""
SELECT COUNT(*) AS Rows_After_Duplicate_Removal
FROM orders_staging
""")

print("Rows After Duplicate Removal:", cursor.fetchone()[0])


Duplicate rows removed: 0
Rows After Duplicate Removal: 9700


In [ ]:
#7.Remove Unwanted spaces
cursor=conn.execute("""
UPDATE orders_staging
SET
  CustomerName=TRIM(CustomerName),
  Email=TRIM(Email),
  Phone=TRIM(Phone),
  City=TRIM(City),
  OrderDate=TRIM(OrderDate),
  Status=TRIM(Status),
  Amount=TRIM(Amount);
  """)
conn.commit()
print("Unwanted spaces removes")
cursor=conn.execute("""
SELECT *
FROM orders_staging
LIMIT 20;
""")
conn.commit()
print("First 20 rows")
rows=cursor.fetchall()
for row in rows:
  print(row)

Unwanted spaces removes
First 20 rows
(3105, 'Jaya Verma', 'jaya.verma801@mail.com', '733-307-7375', 'delhi', '2024-01-05', '3091.09', 'Pending')
(6354, 'Farhan Yadav', 'farhan.yadav65@company.co.in', '23114 80745', 'Bengaluru', '04-14-2023', '1662.89', 'Refunded')
(8690, 'Chitra Reddy', 'chitra.reddy695company.co.in', '379-490-1038', 'pune', '2024-08-03', '1610.32', 'Refunded')
(5858, 'Sneha Iyer', 'sneha.iyer945@company.co.in', '+91 8472931687', 'Coimbatore', '11-08-2023', '2265.32', 'Pending')
(6012, 'kabir kumar', 'kabir.kumar305@outlook.com', '0400977061', 'Coimbatore', '11-19-2024', '2718.23', 'Cancelled')
(2653, 'Kabir Joshi', 'kabir.joshi17@mail.com', '75656 19112', 'Delhi', '02-05-2023', '₹434.17', 'Pending')
(5130, 'Deepa Chatterjee', 'deepa.chatterjee695@outlook.com', '+91 1238943682', 'CHENNAI', '12-02-2024', '2333.70', 'Refunded')
(416, 'Esha Iyer', 'esha.iyer391@gmail.com', '76927 43811', 'HYDERABAD', '02-28-2025', '3818.14', 'Pending')
(3539, 'Arun Yadav', 'arun.yadav764

In [ ]:
#8.Standardize Customer Names
cursor=conn.execute("""
UPDATE orders_staging
SET CustomerName=
  UPPER(SUBSTR(CustomerName,1,1))||
  LOWER(
   SUBSTR(
    CustomerName,
    2,
    INSTR(CustomerName,'')-2
    )
  )||
  ''||
  UPPER(
   SUBSTR(
    CustomerName,
    INSTR(CustomerName,'')+1,
    1
   )
  )||
  LOWER(
   SUBSTR(
    CustomerName,
    INSTR(CustomerName,'')+2
   )
  )
  WHERE
   CustomerName IS NOT NULL
   AND INSTR(CustomerName,'')>0;
  """)
conn.commit()
print("Customer Names Standardized")
cursor=conn.execute("""
SELECT CustomerName
FROM orders_raw
WHERE CustomerName IS NOT NULL
LIMIT 20
""")
print("First 20 Original Customer Names:")
for row in cursor.fetchall():
  print(row[0])

# Standardize Customer Names

def proper_case(name):
    if name is None:
        return None
    return " ".join(str(name).split()).title()

conn.create_function("PROPER_CASE", 1, proper_case)

cursor = conn.execute("""
UPDATE orders_staging
SET CustomerName = (
    SELECT PROPER_CASE(CustomerName)
    FROM orders_raw
    WHERE orders_raw.OrderID = orders_staging.OrderID
    LIMIT 1
)
WHERE EXISTS (
    SELECT 1
    FROM orders_raw
    WHERE orders_raw.OrderID = orders_staging.OrderID
)
AND CustomerName IS NOT NULL
""")

conn.commit()

print("Customer Names Standardized Successfully")

# Display First 20 Names

cursor = conn.execute("""
SELECT CustomerName
FROM orders_staging
WHERE CustomerName IS NOT NULL
AND TRIM(CustomerName) != ''
ORDER BY CustomerName
LIMIT 20
""")

print("\nFirst 20 Standardized Customer Names:")

for row in cursor.fetchall():
    print(row[0])




Customer Names Standardized
First 20 Original Customer Names:
Jaya Verma
Farhan Yadav
  Chitra Reddy 
 Sneha Iyer
  kabir kumar  
Kabir Joshi 
Deepa Chatterjee  
Esha Iyer
Arun Yadav 
 qadir singh  
Wasim Gupta  
Esha Joshi
  Ravi Iyer
HARI KHAN
Manoj Verma 
VIKRAM PILLAI
KABIR MEHTA
  Zoya Das 
Bhavna Pillai 
Omkar Gupta  
Customer Names Standardized Successfully

First 20 Standardized Customer Names:
Anand Bose
Anand Bose
Anand Bose
Anand Bose
Anand Bose
Anand Bose
Anand Bose
Anand Bose
Anand Bose
Anand Bose
Anand Bose
Anand Bose
Anand Bose
Anand Bose
Anand Bose
Anand Chatterjee
Anand Chatterjee
Anand Chatterjee
Anand Chatterjee
Anand Chatterjee


In [ ]:
#9.Standadize City names
cursor=conn.execute("""
UPDATE orders_staging
SET City=
  CASE UPPER(TRIM(City))
    WHEN 'CHENNAI'
     THEN 'Chennai'
    WHEN 'BENGALURU'
     THEN 'Bengaluru'
    WHEN 'COIMBATORE'
     THEN 'Coimbatore'
    WHEN 'MUMBAI'
     THEN 'Mumbai'
    WHEN 'DELHI'
     THEN 'Delhi'
    WHEN 'HYDERABAD'
     THEN 'Hyderabad'
    WHEN 'PUNE'
     THEN 'Pune'
    WHEN 'KOLKATA'
     THEN 'Kolkata'
    ELSE City
  END
WHERE City IS NOT NULL;
""")
conn.commit()
print("City Names Standardized")
print("After TRIM City Names:")
cursor=conn.execute("""
SELECT DISTINCT City
FROM orders_staging
ORDER BY City;
""")
for row in cursor.fetchall():
  print(row[0])




City Names Standardized
After TRIM City Names:
None
Bengaluru
Chennai
Coimbatore
Delhi
Hyderabad
Kolkata
Mumbai
Pune


In [ ]:
#10.Convert Email to Lowercse
cursor=conn.execute("""
UPDATE orders_staging
SET Email=LOWER(TRIM(Email))
WHERE Email IS NOT NULL;
""")
conn.commit()
print("Email Converted to Lowercase")
print("First 20 Emails:")
cursor=conn.execute("""
SELECT Email
FROM orders_staging
WHERE Email IS NOT NULL
LIMIT 20;
""")
for row in cursor.fetchall():
  print(row[0])


Email Converted to Lowercase
First 20 Emails:
jaya.verma801@mail.com
farhan.yadav65@company.co.in
chitra.reddy695company.co.in
sneha.iyer945@company.co.in
kabir.kumar305@outlook.com
kabir.joshi17@mail.com
deepa.chatterjee695@outlook.com
esha.iyer391@gmail.com
arun.yadav764@mail.com
qadir.singh378@mail.com
wasim.gupta712@mail.com
esha.joshi714@yahoo.com
ravi.iyer314@outlook.com
hari.khan234@yahoo.com
manoj.verma369@gmail.com
vikram.pillai568@company.co.in
kabir.mehta681@yahoo.com
zoya.das723@mail
bhavna.pillai560@gmail.com
omkar.gupta615@outlook.com


In [ ]:
#11.Stndardize Status
cursor=conn.execute("""
UPDATE orders_staging
SET Status=
  CASE UPPER(TRIM(Status))
    WHEN 'COMPLETED'
     THEN 'Completed'
    WHEN 'CANCELLED'
     THEN 'Cancelled'
    WHEN 'PENDING'
     THEN 'Pending'
    WHEN 'REFUNDED'
     THEN 'Refunded'
    WHEN 'N/A'
     THEN NULL
    WHEN 'NA'
     THEN NULL
    ELSE Status
  END
WHERE STATUS IS NOT NULL;
""")
conn.commit()
print("Status Standardized")
cursor=conn.execute(
    """
    SELECT
        Status,
        COUNT(*) AS Total
    FROM orders_staging
    GROUP BY Status
    ORDER BY Total DESC;
""")
for row in cursor.fetchall():
  print(row)


Status Standardized
('Refunded', 2311)
('Pending', 2182)
('Completed', 2180)
('Cancelled', 2164)
(None, 804)
('Canceled', 42)
('CANCELED', 13)
('canceled', 4)


In [ ]:
# 12 Handle missing values
cursor=conn.execute("""
UPDATE orders_staging
SET CustomerName='Unknown Customer'
WHERE CustomerName IS NULL
  OR TRIM(CustomerName)='';
""")
cursor=conn.execute("""
UPDATE orders_staging
SET City='Unknown'
WHERE City IS NULL
  OR TRIM(CIty)='';
""")
cursor=conn.execute("""
UPDATE orders_staging
SET Status='Unknown'
WHERE Status IS NULL
  OR TRIM(Status)='';
""")
print("Missing Values Handled Successfully")
cursor=conn.execute("""
SELECT CustomerName,City,Status
FROM orders_staging
LIMIT 20
""")
print("Total Records Displayed",len(rows))
for row in cursor.fetchall():
  print(row)


Missing Values Handled Successfully
Total Records Displayed 20
('Jaya Verma', 'Delhi', 'Pending')
('Farhan Yadav', 'Bengaluru', 'Refunded')
('Chitra Reddy', 'Pune', 'Refunded')
('Sneha Iyer', 'Coimbatore', 'Pending')
('Kabir Kumar', 'Coimbatore', 'Cancelled')
('Kabir Joshi', 'Delhi', 'Pending')
('Deepa Chatterjee', 'Chennai', 'Refunded')
('Esha Iyer', 'Hyderabad', 'Pending')
('Arun Yadav', 'Mumbai', 'Completed')
('Qadir Singh', 'Delhi', 'Pending')
('Wasim Gupta', 'Chennai', 'Completed')
('Esha Joshi', 'Chennai', 'Refunded')
('Ravi Iyer', 'Mumbai', 'Completed')
('Hari Khan', 'Chennai', 'Cancelled')
('Manoj Verma', 'Unknown', 'Pending')
('Vikram Pillai', 'Hyderabad', 'Unknown')
('Kabir Mehta', 'Bengaluru', 'Completed')
('Zoya Das', 'Chennai', 'Refunded')
('Bhavna Pillai', 'Delhi', 'Completed')
('Omkar Gupta', 'Coimbatore', 'Pending')


In [ ]:

# 13.Standardize OrderDate

cursor = conn.execute("""
UPDATE orders_staging
SET OrderDate =
    CASE
        WHEN TRIM(OrderDate) LIKE '____-__-__'
            THEN TRIM(OrderDate)

        WHEN TRIM(OrderDate) LIKE '__-__-____'
            THEN SUBSTR(TRIM(OrderDate), 7, 4) || '-' ||
                 SUBSTR(TRIM(OrderDate), 1, 2) || '-' ||
                 SUBSTR(TRIM(OrderDate), 4, 2)

        WHEN TRIM(OrderDate) LIKE '__/__/____'
            THEN SUBSTR(TRIM(OrderDate), 7, 4) || '-' ||
                 SUBSTR(TRIM(OrderDate), 4, 2) || '-' ||
                 SUBSTR(TRIM(OrderDate), 1, 2)

        ELSE TRIM(OrderDate)
    END
WHERE OrderDate IS NOT NULL
""")

conn.commit()

print("Order Date Standardized Successfully")

# Display First 20 Records

cursor = conn.execute("""
SELECT OrderID, OrderDate
FROM orders_staging
LIMIT 20
""")

print("\nFirst 20 Standardized Order Dates:")

for row in cursor.fetchall():
    print(row)


Order Date Standardized Successfully

First 20 Standardized Order Dates:
(3105, '2024-01-05')
(6354, '2023-04-14')
(8690, '2024-08-03')
(5858, '2023-11-08')
(6012, '2024-11-19')
(2653, '2023-02-05')
(5130, '2024-12-02')
(416, '2025-02-28')
(3539, '2025-02-05')
(1704, '2025-07-24')
(3528, '2023-03-08')
(7900, '2025-06-20')
(3027, '2024-08-14')
(747, '2023-06-27')
(2818, '2025-03-19')
(1916, '2024-07-17')
(6985, '2025-09-24')
(3328, '2025-05-03')
(7954, '2025-02-01')
(1770, '2023-10-08')


In [22]:

# 14.Validate Email Addresses

cursor = conn.execute("""
UPDATE orders_staging
SET Email_Valid =
    CASE
        WHEN Email IS NULL OR TRIM(Email) = '' THEN 0
        WHEN LOWER(TRIM(Email)) LIKE '%_@_%._%'
             AND TRIM(Email) NOT LIKE '% %'
        THEN 1
        ELSE 0
    END
""")

conn.commit()

print("Email Validation Completed")

# Display First 20 Records

cursor = conn.execute("""
SELECT OrderID, CustomerName, Email, Email_Valid
FROM orders_staging
LIMIT 20
""")

for row in cursor.fetchall():
    print(row)


Email Validation Completed
(3105, 'Jaya Verma', 'jaya.verma801@mail.com', 1)
(6354, 'Farhan Yadav', 'farhan.yadav65@company.co.in', 1)
(8690, '  Chitra Reddy ', 'chitra.reddy695company.co.in', 0)
(5858, ' Sneha Iyer', 'sneha.iyer945@company.co.in', 1)
(6012, '  kabir kumar  ', 'kabir.kumar305@outlook.com', 1)
(2653, 'Kabir Joshi ', 'kabir.joshi17@mail.com', 1)
(5130, 'Deepa Chatterjee  ', 'deepa.chatterjee695@outlook.com', 1)
(416, 'Esha Iyer', 'esha.iyer391@gmail.com', 1)
(3539, 'Arun Yadav ', 'arun.yadav764@mail.com', 1)
(1704, ' qadir singh  ', 'qadir.singh378@mail.com', 1)
(3528, 'Wasim Gupta  ', 'wasim.gupta712@mail.com', 1)
(7900, 'Esha Joshi', 'esha.joshi714@yahoo.com', 1)
(3027, '  Ravi Iyer', 'ravi.iyer314@outlook.com', 1)
(747, 'HARI KHAN', 'hari.khan234@yahoo.com', 1)
(2818, 'Manoj Verma ', 'manoj.verma369@gmail.com', 1)
(1916, 'VIKRAM PILLAI', 'vikram.pillai568@company.co.in', 1)
(6985, 'KABIR MEHTA', 'kabir.mehta681@yahoo.com', 1)
(3328, '  Zoya Das ', 'zoya.das723@mail', 

In [25]:

#15. Clean Phone Numbers

cursor = conn.cursor()

cursor.execute("""
UPDATE orders_staging
SET Phone =
    REPLACE(
        REPLACE(
            REPLACE(
                REPLACE(
                    REPLACE(Phone, '-', ''),
                ' ', ''),
            '+', ''),
        '(', ''),
    ')', '')
WHERE Phone IS NOT NULL;
""")

conn.commit()

print("Phone Numbers Cleaned Successfully")

# Display First 20 Records

cursor.execute("""
SELECT OrderID, CustomerName, Phone
FROM orders_staging
LIMIT 20;
""")

for row in cursor.fetchall():
    print(row)


Phone Numbers Cleaned Successfully
(3105, 'Jaya Verma', '7333077375')
(6354, 'Farhan Yadav', '2311480745')
(8690, '  Chitra Reddy ', '3794901038')
(5858, ' Sneha Iyer', '918472931687')
(6012, '  kabir kumar  ', '0400977061')
(2653, 'Kabir Joshi ', '7565619112')
(5130, 'Deepa Chatterjee  ', '911238943682')
(416, 'Esha Iyer', '7692743811')
(3539, 'Arun Yadav ', '914798431264')
(1704, ' qadir singh  ', '4311621065')
(3528, 'Wasim Gupta  ', '1625983913')
(7900, 'Esha Joshi', '5205227568')
(3027, '  Ravi Iyer', '6269595197')
(747, 'HARI KHAN', '4423471998')
(2818, 'Manoj Verma ', '7293060898')
(1916, 'VIKRAM PILLAI', '6269877351')
(6985, 'KABIR MEHTA', '2954145386')
(3328, '  Zoya Das ', '8485505635')
(7954, 'Bhavna Pillai ', '5700090038')
(1770, 'Omkar Gupta  ', '6738007506')


In [28]:
#16.Remove +91 country code
cursor=conn.execute("""
UPDATE orders_staging
SET Phone=SUBSTR(Phone,3)
WHERE
  Phone IS NOT NULL
  AND LENGTH(Phone)=12
  AND SUBSTR(Phone,1,2)='91';
  """)
conn.commit()
print("Country Code Removed")
cursor=conn.execute("""
SELECT OrderID,CustomerName,Phone
FROM orders_staging
LIMIT 20;
""")
for row in cursor.fetchall():
  print(row)


Country Code Removed
(3105, 'Jaya Verma', '7333077375')
(6354, 'Farhan Yadav', '2311480745')
(8690, '  Chitra Reddy ', '3794901038')
(5858, ' Sneha Iyer', '8472931687')
(6012, '  kabir kumar  ', '0400977061')
(2653, 'Kabir Joshi ', '7565619112')
(5130, 'Deepa Chatterjee  ', '1238943682')
(416, 'Esha Iyer', '7692743811')
(3539, 'Arun Yadav ', '4798431264')
(1704, ' qadir singh  ', '4311621065')
(3528, 'Wasim Gupta  ', '1625983913')
(7900, 'Esha Joshi', '5205227568')
(3027, '  Ravi Iyer', '6269595197')
(747, 'HARI KHAN', '4423471998')
(2818, 'Manoj Verma ', '7293060898')
(1916, 'VIKRAM PILLAI', '6269877351')
(6985, 'KABIR MEHTA', '2954145386')
(3328, '  Zoya Das ', '8485505635')
(7954, 'Bhavna Pillai ', '5700090038')
(1770, 'Omkar Gupta  ', '6738007506')


In [33]:
# 17.Validate Phone Numbers

cursor = conn.execute("""
UPDATE orders_staging
SET Phone_Valid =
    CASE
        WHEN Phone IS NOT NULL
        AND LENGTH(REPLACE(REPLACE(REPLACE(Phone, '-', ''), ' ', ''), '+', '')) >= 10
        THEN 1
        ELSE 0
    END
""")

conn.commit()

print("Phone Validation Completed")

cursor = conn.execute("""
SELECT Phone, Phone_Valid
FROM orders_staging
LIMIT 20
""")

for row in cursor.fetchall():
    print(row)

Phone Validation Completed
('7333077375', 1)
('2311480745', 1)
('3794901038', 1)
('8472931687', 1)
('0400977061', 1)
('7565619112', 1)
('1238943682', 1)
('7692743811', 1)
('4798431264', 1)
('4311621065', 1)
('1625983913', 1)
('5205227568', 1)
('6269595197', 1)
('4423471998', 1)
('7293060898', 1)
('6269877351', 1)
('2954145386', 1)
('8485505635', 1)
('5700090038', 1)
('6738007506', 1)


In [37]:
#18. Clean Amount

cursor = conn.execute("""
UPDATE orders_staging
SET Amount = REPLACE(
    REPLACE(
        REPLACE(
            REPLACE(Amount, '$', ''),
        ',', ''),
    ' ', ''),
'₹', '')
WHERE Amount IS NOT NULL
""")

conn.commit()

print("Amount Cleaning Completed")

cursor = conn.execute("""
SELECT OrderID, Amount
FROM orders_staging
LIMIT 20
""")
cursor = conn.execute("""
SELECT OrderID, Amount
FROM orders_staging
WHERE Amount IS NOT NULL
AND TRIM(Amount) != ''
LIMIT 20
""")

print("First 20 Amount Records:")

for row in cursor.fetchall():
    print(row)

Amount Cleaning Completed
First 20 Amount Records:
(3105, '3091.09')
(6354, '1662.89')
(8690, '1610.32')
(5858, '2265.32')
(6012, '2718.23')
(2653, '434.17')
(5130, '2333.70')
(416, '3818.14')
(3539, '1297.01')
(1704, '2274.49')
(3528, '1928.67')
(7900, '2584.67')
(3027, '3557.42')
(747, '3198.56')
(2818, '3910.42')
(1916, '4958.93')
(6985, '1405.41')
(7954, '3580.75')
(1770, '2729.8')
(7754, '4632.3')


In [42]:
#19. Create Numeric Amount Column

cursor = conn.execute("""
UPDATE orders_staging
SET Amount_Clean = CAST(
    REPLACE(
        REPLACE(
            REPLACE(
                REPLACE(Amount, '$', ''),
                ',', ''
            ),
            '₹', ''
        ),
        ' ', ''
    ) AS REAL
)
WHERE Amount IS NOT NULL
AND TRIM(Amount) != ''
""")

conn.commit()

print("Numeric Amount Column Updated Successfully")


# Display first 20 valid Amount records

cursor = conn.execute("""
SELECT
    OrderID,
    Amount,
    Amount_Clean
FROM orders_staging
WHERE Amount_Clean IS NOT NULL
AND Amount_Clean != 0
LIMIT 20
""")

rows = cursor.fetchall()

print("Numeric Amount - First 20 Records")

for row in rows:
    print(row)

Numeric Amount Column Updated Successfully
Numeric Amount - First 20 Records
(3105, '3091.09', 3091.09)
(6354, '1662.89', 1662.89)
(8690, '1610.32', 1610.32)
(5858, '2265.32', 2265.32)
(6012, '2718.23', 2718.23)
(2653, '434.17', 434.17)
(5130, '2333.70', 2333.7)
(416, '3818.14', 3818.14)
(3539, '1297.01', 1297.01)
(1704, '2274.49', 2274.49)
(3528, '1928.67', 1928.67)
(7900, '2584.67', 2584.67)
(3027, '3557.42', 3557.42)
(747, '3198.56', 3198.56)
(2818, '3910.42', 3910.42)
(1916, '4958.93', 4958.93)
(6985, '1405.41', 1405.41)
(7954, '3580.75', 3580.75)
(1770, '2729.8', 2729.8)
(7754, '4632.3', 4632.3)


In [45]:

# 20.Create Amount_Flag Column

cursor = conn.cursor()

cursor.execute("PRAGMA table_info(orders_staging)")
columns = [row[1] for row in cursor.fetchall()]

if "Amount_Flag" not in columns:
    cursor.execute("""
    ALTER TABLE orders_staging
    ADD COLUMN Amount_Flag TEXT
    """)

# Classify Amount Values

cursor.execute("""
UPDATE orders_staging
SET Amount_Flag =
    CASE
        WHEN Amount_Clean IS NULL
            THEN 'Missing'
        WHEN Amount_Clean < 0
            THEN 'Negative'
        WHEN Amount_Clean > 20000
            THEN 'Outlier'
        ELSE 'OK'
    END
""")

conn.commit()

print("Amount Flag Created Successfully")

# Display First 20 Records

cursor.execute("""
SELECT OrderID, Amount_Clean, Amount_Flag
FROM orders_staging
LIMIT 20
""")

for row in cursor.fetchall():
    print(row)
cursor.execute("""
UPDATE orders_staging
SET Amount_Clean = 'Missing'
WHERE Amount_Clean IS NULL
""")

conn.commit()
print("Missing Amount Updated")



Amount Flag Created Successfully
(3105, 3091.09, 'OK')
(6354, 1662.89, 'OK')
(8690, 1610.32, 'OK')
(5858, 2265.32, 'OK')
(6012, 2718.23, 'OK')
(2653, 434.17, 'OK')
(5130, 2333.7, 'OK')
(416, 3818.14, 'OK')
(3539, 1297.01, 'OK')
(1704, 2274.49, 'OK')
(3528, 1928.67, 'OK')
(7900, 2584.67, 'OK')
(3027, 3557.42, 'OK')
(747, 3198.56, 'OK')
(2818, 3910.42, 'OK')
(1916, 4958.93, 'OK')
(6985, 1405.41, 'OK')
(3328, 'Missing', 'Outlier')
(7954, 3580.75, 'OK')
(1770, 2729.8, 'OK')
Missing Amount Updated


In [47]:
#Create final clean table
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS orders_clean")

cursor.execute("""
CREATE TABLE orders_clean AS
SELECT
    OrderID,
    CustomerName,
    Email,
    Email_Valid,
    Phone,
    Phone_Valid,
    City,
    OrderDate,
    Amount_Clean AS Amount,
    Amount_Flag,
    Status
FROM orders_staging
""")

conn.commit()

print("Final Clean Table Created Successfully")
cursor.execute("""
SELECT *
FROM orders_clean
LIMIT 20
""")

for row in cursor.fetchall():
    print(row)

Final Clean Table Created Successfully
(3105, 'Jaya Verma', 'jaya.verma801@mail.com', 1, '7333077375', 1, ' delhi', '2024-01-05', 3091.09, 'OK', 'Pending')
(6354, 'Farhan Yadav', 'farhan.yadav65@company.co.in', 1, '2311480745', 1, 'Bengaluru', '04-14-2023', 1662.89, 'OK', 'Refunded')
(8690, '  Chitra Reddy ', 'chitra.reddy695company.co.in', 0, '3794901038', 1, ' pune', '2024-08-03', 1610.32, 'OK', 'Refunded')
(5858, ' Sneha Iyer', 'sneha.iyer945@company.co.in', 1, '8472931687', 1, ' Coimbatore  ', '11-08-2023', 2265.32, 'OK', 'Pending')
(6012, '  kabir kumar  ', 'kabir.kumar305@outlook.com', 1, '0400977061', 1, ' Coimbatore ', '11-19-2024', 2718.23, 'OK', 'Cancelled')
(2653, 'Kabir Joshi ', 'kabir.joshi17@mail.com', 1, '7565619112', 1, '  Delhi', '02-05-2023', 434.17, 'OK', 'Pending')
(5130, 'Deepa Chatterjee  ', 'deepa.chatterjee695@outlook.com', 1, '1238943682', 1, 'CHENNAI ', '12-02-2024', 2333.7, 'OK', 'Refunded')
(416, 'Esha Iyer', 'esha.iyer391@gmail.com', 1, '7692743811', 1, ' H

In [48]:
#22.Average order amount by city
cursor = conn.cursor()

cursor.execute("""
SELECT
    City,
    ROUND(AVG(Amount), 2) AS Average_Amount
FROM orders_clean
WHERE Amount_Flag = 'OK'
GROUP BY City
ORDER BY Average_Amount DESC
""")

print("City\tAverage Amount")
print("-" * 35)

for row in cursor.fetchall():
    print(row[0], "\t", row[1])

City	Average Amount
-----------------------------------
  chennai  	 3763.13
 MUMBAI  	 3518.07
  CHENNAI  	 3333.55
 chennai 	 3297.02
 PUNE   	 3208.27
 BENGALURU  	 3157.01
 COIMBATORE 	 3149.88
  BENGALURU   	 2963.6
  Mumbai  	 2949.54
  bengaluru   	 2853.61
  HYDERABAD   	 2852.24
 kolkata   	 2844.0
hyderabad  	 2820.69
mumbai  	 2818.76
  COIMBATORE   	 2817.74
  bengaluru 	 2810.92
 mumbai   	 2805.88
  DELHI 	 2769.41
  COIMBATORE  	 2745.99
 Bengaluru  	 2745.9
  kolkata 	 2738.57
BENGALURU   	 2727.18
 kolkata 	 2726.8
kolkata 	 2720.9
  Coimbatore 	 2719.53
KOLKATA  	 2711.65
  mumbai  	 2684.87
coimbatore 	 2670.75
kolkata   	 2669.36
CHENNAI  	 2660.62
 hyderabad   	 2655.02
  coimbatore 	 2645.68
  Pune  	 2645.41
 BENGALURU   	 2645.24
 Bengaluru 	 2642.25
delhi  	 2639.79
Mumbai 	 2632.25
  Pune   	 2631.64
PUNE 	 2626.79
Pune   	 2625.53
  KOLKATA  	 2625.02
Pune 	 2621.69
  Delhi 	 2613.46
  DELHI   	 2602.53
 PUNE  	 2601.75
COIMBATORE  	 2592.23
Hyderabad  	 2588

In [49]:
23.#Total sales by city
cursor = conn.cursor()

cursor.execute("""
SELECT
    City,
    ROUND(SUM(Amount), 2) AS Total_Sales
FROM orders_clean
WHERE Amount_Flag = 'OK'
GROUP BY City
ORDER BY Total_Sales DESC
""")

print("City\tTotal Sales")
print("-" * 35)

for row in cursor.fetchall():
    print(row[0], "\t", row[1])

City	Total Sales
-----------------------------------
None 	 2292545.8
Mumbai 	 594889.37
Kolkata 	 560676.59
Pune 	 547932.18
Chennai 	 529842.98
Coimbatore 	 511222.82
Bengaluru 	 506938.84
Delhi 	 494241.28
Hyderabad 	 456769.18
COIMBATORE 	 256076.12
CHENNAI 	 233902.61
coimbatore 	 232355.14
PUNE 	 228530.56
KOLKATA 	 226112.56
kolkata 	 223113.45
bengaluru 	 211037.94
  Coimbatore 	 209404.02
 Bengaluru 	 206095.19
Delhi   	 205760.8
DELHI 	 205698.0
  Delhi 	 203849.86
Coimbatore  	 202966.79
  Mumbai 	 200861.37
MUMBAI 	 198822.66
 Kolkata 	 192921.82
delhi 	 189566.4
Pune   	 189038.27
Pune  	 187867.25
pune 	 187107.42
Hyderabad  	 186356.51
Kolkata   	 183922.87
chennai 	 182633.91
  Bengaluru 	 182488.55
 Pune 	 181057.43
BENGALURU 	 176472.04
 Delhi 	 175347.67
mumbai 	 175309.89
hyderabad 	 169766.44
Chennai  	 168088.65
Delhi  	 167675.25
Coimbatore   	 163993.78
Hyderabad   	 163022.74
 Coimbatore 	 162791.13
 Mumbai 	 162141.82
HYDERABAD 	 160426.82
  Hyderabad 	 158785

In [50]:
#Step 24 – Number of Orders by City.
cursor = conn.cursor()

cursor.execute("""
SELECT
    City,
    COUNT(*) AS Total_Orders
FROM orders_clean
GROUP BY City
ORDER BY Total_Orders DESC
""")

print("City\tTotal Orders")
print("-" * 35)

for row in cursor.fetchall():
    print(row[0], "\t", row[1])

City	Total Orders
-----------------------------------
None 	 1011
Kolkata 	 243
Mumbai 	 233
Delhi 	 228
Coimbatore 	 228
Pune 	 225
Chennai 	 220
Bengaluru 	 220
Hyderabad 	 198
COIMBATORE 	 105
MUMBAI 	 97
CHENNAI 	 96
KOLKATA 	 94
coimbatore 	 93
DELHI 	 93
PUNE 	 91
Delhi   	 91
Coimbatore  	 89
  Mumbai 	 89
pune 	 87
kolkata 	 87
bengaluru 	 87
chennai 	 86
mumbai 	 85
delhi 	 85
 Bengaluru 	 84
 Kolkata 	 81
  Coimbatore 	 81
Kolkata   	 80
hyderabad 	 79
 Pune 	 79
  Delhi 	 78
Pune  	 77
HYDERABAD 	 77
Mumbai  	 76
Chennai  	 76
Hyderabad  	 75
Hyderabad   	 74
BENGALURU 	 74
 Delhi 	 74
 Coimbatore 	 74
Pune   	 73
Bengaluru  	 73
  Chennai 	 73
  Bengaluru 	 73
  Hyderabad 	 72
Bengaluru   	 71
 Chennai 	 71
Kolkata  	 69
Delhi  	 69
Chennai   	 69
Mumbai   	 68
 Mumbai 	 67
Coimbatore   	 65
 Hyderabad 	 64
  Kolkata 	 61
  Pune 	 59
 CHENNAI 	 41
  PUNE 	 40
  CHENNAI 	 39
COIMBATORE   	 38
PUNE  	 37
  pune 	 37
  bengaluru 	 37
  COIMBATORE 	 37
COIMBATORE  	 36
chennai 

In [51]:
#25.Completed orders
cursor = conn.cursor()

cursor.execute("""
SELECT *
FROM orders_clean
WHERE Status = 'Completed'
LIMIT 20
""")

for row in cursor.fetchall():
    print(row)

(3539, 'Arun Yadav ', 'arun.yadav764@mail.com', 1, '4798431264', 1, ' Mumbai  ', '02-05-2025', 1297.01, 'OK', 'Completed')
(3528, 'Wasim Gupta  ', 'wasim.gupta712@mail.com', 1, '1625983913', 1, 'Chennai', '08/03/2023', 1928.67, 'OK', 'Completed')
(3027, '  Ravi Iyer', 'ravi.iyer314@outlook.com', 1, '6269595197', 1, '  Mumbai  ', '14/08/2024', 3557.42, 'OK', 'Completed')
(6985, 'KABIR MEHTA', 'kabir.mehta681@yahoo.com', 1, '2954145386', 1, 'bengaluru', '09-24-2025', 1405.41, 'OK', 'Completed')
(7954, 'Bhavna Pillai ', 'bhavna.pillai560@gmail.com', 1, '5700090038', 1, 'DELHI  ', '2025-02-01', 3580.75, 'OK', 'Completed')
(7754, 'ANAND KHAN', None, 0, '5886491802', 1, 'KOLKATA', '02/06/2023', 4632.3, 'OK', 'Completed')
(7026, 'zoya chatterjee ', 'zoya.chatterjee112@company.co.in', 1, '4618236645', 1, ' Hyderabad  ', '11/04/2024', 2542.14, 'OK', 'Completed')
(8839, 'Tarun Verma', 'tarun.verma39@outlook.com', 1, '4471064873', 1, 'CHENNAI', '2024-03-07', 1756.89, 'OK', 'Completed')
(6828, 'MA

In [52]:
#Part 2 – Count Completed Orders
cursor.execute("""
SELECT
    COUNT(*) AS Completed_Orders
FROM orders_clean
WHERE Status = 'Completed'
""")

result = cursor.fetchone()

print("Total Completed Orders:", result[0])

Total Completed Orders: 1539


In [53]:
#Step 26 – Cancelled Orders by City.
cursor = conn.cursor()

cursor.execute("""
SELECT
    City,
    COUNT(*) AS Cancelled_Orders
FROM orders_clean
WHERE Status = 'Cancelled'
GROUP BY City
ORDER BY Cancelled_Orders DESC
""")

print("City\tCancelled Orders")
print("-" * 40)

for row in cursor.fetchall():
    print(row[0], "\t", row[1])

City	Cancelled Orders
----------------------------------------
None 	 178
Kolkata 	 44
Chennai 	 42
Mumbai 	 36
Delhi 	 35
Coimbatore 	 34
Bengaluru 	 34
Pune 	 31
Hyderabad 	 28
CHENNAI 	 24
chennai 	 20
 Kolkata 	 19
 Coimbatore 	 19
  Coimbatore 	 19
  Mumbai 	 18
Delhi   	 17
DELHI 	 17
MUMBAI 	 16
Chennai   	 16
 Delhi 	 16
  Pune 	 16
coimbatore 	 15
COIMBATORE 	 15
  Kolkata 	 15
hyderabad 	 14
 Pune 	 14
Pune  	 13
PUNE 	 13
Mumbai  	 13
Kolkata   	 13
KOLKATA 	 13
Hyderabad   	 13
  PUNE 	 13
mumbai 	 12
bengaluru 	 12
HYDERABAD 	 12
Bengaluru   	 12
 Chennai 	 12
  Chennai 	 12
  Bengaluru 	 12
kolkata 	 11
delhi 	 11
Kolkata  	 11
Coimbatore  	 11
Chennai  	 11
BENGALURU 	 11
Mumbai   	 10
  Hyderabad 	 10
Hyderabad  	 9
COIMBATORE   	 9
Bengaluru  	 9
 Bengaluru 	 9
pune 	 8
mumbai   	 8
hyderabad   	 8
 coimbatore 	 8
  mumbai 	 8
  HYDERABAD 	 8
  Delhi 	 8
HYDERABAD   	 7
Coimbatore   	 7
 Mumbai 	 7
 Hyderabad  	 7
 Chennai   	 7
  Mumbai   	 7
  KOLKATA 	 7
  Chennai  

In [54]:
#Step 27 – Negative Amount Orders.
cursor = conn.cursor()

cursor.execute("""
SELECT
    OrderID,
    CustomerName,
    Amount,
    Status
FROM orders_clean
WHERE Amount_Flag = 'Negative'
ORDER BY Amount
""")

print("OrderID | CustomerName | Amount | Status")
print("-" * 60)

for row in cursor.fetchall():
    print(row)

OrderID | CustomerName | Amount | Status
------------------------------------------------------------
(2990, 'NISHA VERMA', -872567.0, 'CANCELLED  ')
(4478, ' Nisha Gupta ', -835860.0, 'Cancelled')
(9574, '  Arun Kumar', -719801.0, None)
(7482, '  FARHAN KAPOOR', -717485.0, '  CANCELLED')
(660, 'HARI YADAV  ', -706542.0, 'Cancelled')
(2854, 'Gowri Khan  ', -703265.0, 'Completed')
(2056, 'Gowri Mehta  ', -677974.0, '  COMPLETED')
(8287, 'BALA CHATTERJEE  ', -651337.0, None)
(2661, '  HARI RAO', -636841.0, 'Completed')
(713, 'farhan gupta', -634196.0, '  REFUNDED ')
(6637, 'ARUN IYER', -583427.0, 'Pending')
(2550, 'GOWRI CHATTERJEE  ', -557613.0, 'Pending')
(572, ' Qadir Rao', -545696.0, 'refunded')
(6642, 'HARI DAS', -536351.0, '  CANCELLED')
(3280, 'priya iyer ', -526671.0, '  pending  ')
(8791, '  bhavna bose', -512628.0, 'Completed')
(5230, 'zoya rao', -500425.0, 'Completed')
(4813, 'Isha Menon', -495650.0, 'Completed')
(9505, '  Manoj Nair', -492752.0, 'Completed')
(3456, 'vikram sh

In [55]:
#Step 28 – Monthly Sales Analysis
cursor = conn.cursor()

cursor.execute("""
SELECT
    SUBSTR(OrderDate, 1, 7) AS Year_Month,
    COUNT(*) AS Total_Orders,
    ROUND(SUM(Amount), 2) AS Total_Sales
FROM orders_clean
WHERE
    Amount_Flag = 'OK'
    AND OrderDate IS NOT NULL
GROUP BY SUBSTR(OrderDate, 1, 7)
ORDER BY Year_Month
""")

print("Year-Month | Total Orders | Total Sales")
print("-" * 50)

for row in cursor.fetchall():
    print(row[0], "|", row[1], "|", row[2])

Year-Month | Total Orders | Total Sales
--------------------------------------------------
01-01-2 | 8 | 19442.57
01-02-2 | 6 | 15086.19
01-03-2 | 10 | 40679.97
01-04-2 | 9 | 17051.14
01-05-2 | 7 | 14182.79
01-06-2 | 6 | 11991.91
01-07-2 | 4 | 11732.55
01-08-2 | 11 | 21800.89
01-09-2 | 6 | 16827.19
01-10-2 | 10 | 16528.89
01-11-2 | 8 | 21235.44
01-12-2 | 6 | 13817.75
01-13-2 | 14 | 30655.33
01-14-2 | 14 | 30149.01
01-15-2 | 11 | 26434.14
01-16-2 | 11 | 24189.84
01-17-2 | 13 | 18253.16
01-18-2 | 7 | 15883.44
01-19-2 | 9 | 20094.15
01-20-2 | 6 | 13646.15
01-21-2 | 10 | 22677.17
01-22-2 | 4 | 6797.83
01-23-2 | 7 | 10196.63
01-24-2 | 9 | 20473.26
01-25-2 | 6 | 13162.97
01-26-2 | 12 | 29886.63
01-27-2 | 15 | 40165.6
01-28-2 | 8 | 10604.73
01/01/2 | 8 | 17892.54
01/02/2 | 9 | 24036.25
01/03/2 | 6 | 13752.58
01/04/2 | 10 | 24121.15
01/05/2 | 10 | 30926.87
01/06/2 | 9 | 19787.76
01/07/2 | 9 | 17564.99
01/08/2 | 7 | 10790.27
01/09/2 | 10 | 18396.51
01/10/2 | 8 | 20076.07
01/11/2 | 13 | 42795.49

In [56]:
import pandas as pd

df_clean = pd.read_sql_query(
    "SELECT * FROM orders_clean",
    conn
)

print(df_clean.head(20))

    OrderID        CustomerName                            Email  Email_Valid  \
0      3105          Jaya Verma           jaya.verma801@mail.com            1   
1      6354        Farhan Yadav     farhan.yadav65@company.co.in            1   
2      8690       Chitra Reddy      chitra.reddy695company.co.in            0   
3      5858          Sneha Iyer      sneha.iyer945@company.co.in            1   
4      6012       kabir kumar         kabir.kumar305@outlook.com            1   
5      2653        Kabir Joshi            kabir.joshi17@mail.com            1   
6      5130  Deepa Chatterjee    deepa.chatterjee695@outlook.com            1   
7       416           Esha Iyer           esha.iyer391@gmail.com            1   
8      3539         Arun Yadav            arun.yadav764@mail.com            1   
9      1704       qadir singh            qadir.singh378@mail.com            1   
10     3528       Wasim Gupta            wasim.gupta712@mail.com            1   
11     7900          Esha Jo

In [57]:
df_clean.to_excel("orders_clean.xlsx", index=False)

print("Excel File Created Successfully")

Excel File Created Successfully


In [58]:
from google.colab import files

files.download("orders_clean.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>